## Create a MetaClassifier based on LLM Outputs

I had only fed the article full-text to the different LLMs to get their output. However, in the following, it shows that there is some data limitations as article text are not fully-cleaned. So, the results are not great.

To augment this, I will also combined with the cosine similarity scores using an embedding model and feed it into the meta classifier together.

In [1]:
import numpy as np
import pandas as pd
import re
import json
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score


class EnsembleMetaClassifier:
    def __init__(self, base_model_names=None, meta_model=None):
        """
        base_model_names: Optional list of strings representing base model names for tracking/logging.
        meta_model: The scikit-learn compatible classifier to be used as the meta model. Defaults to LogisticRegression.
        """
        self.base_model_names = base_model_names or []
        self.meta_model = meta_model or LogisticRegression(max_iter=1000)
        self.best_params = None
        self.fitted = False

    def _get_model_and_grid(self, model_type):
        """
        Given a model type, returns the model and associated hyperparameter grid.
        """
        if model_type == "logistic":
            model = LogisticRegression(max_iter=1000)
            param_grid = {
                "C": [0.01, 0.1, 1, 10],
                "penalty": ["l2"],
                "solver": ["lbfgs"]
            }
        elif model_type == "random_forest":
            model = RandomForestClassifier()
            param_grid = {
                "n_estimators": [50, 100, 200],
                "max_depth": [None, 10, 20],
                "min_samples_split": [2, 5]
            }
        elif model_type == "xgboost":
            model = XGBClassifier(use_label_encoder=False, eval_metric="logloss")
            param_grid = {
                "n_estimators": [50, 100, 200],
                "max_depth": [3, 6, 10],
                "learning_rate": [0.01, 0.1, 0.2],
            }
        else:
            raise ValueError(f"Unsupported model type: {model_type}")

        return model, param_grid

    def _create_search(self, model, param_grid, search_type, scoring, cv, n_iter, bayes_trials):
        """
        Creates the appropriate hyperparameter search object.
        """
        if search_type == "grid":
            return GridSearchCV(model, param_grid, scoring=scoring, cv=cv)

        elif search_type == "random":
            return RandomizedSearchCV(model, param_distributions=param_grid, scoring=scoring, cv=cv, n_iter=n_iter)

        elif search_type == "bayesian":
            try:
                from skopt import BayesSearchCV
                from skopt.space import Real, Integer, Categorical
            except ImportError:
                raise ImportError("Please install scikit-optimize: pip install scikit-optimize")

            def convert_grid(grid):
                space = {}
                for k, v in grid.items():
                    if all(isinstance(i, int) for i in v):
                        space[k] = Integer(min(v), max(v))
                    elif all(isinstance(i, float) for i in v):
                        space[k] = Real(min(v), max(v))
                    else:
                        space[k] = Categorical(v)
                return space

            search_space = convert_grid(param_grid)
            return BayesSearchCV(model, search_space, scoring=scoring, cv=cv, n_iter=bayes_trials, n_jobs=-1)

        else:
            raise ValueError(f"Unknown search type: {search_type}")

    def fit(self, X_preds, y_true, model_type="logistic", search_type="grid", scoring="f1", cv=3, n_iter=20, bayes_trials=30):
        """
        Fits the meta model using predictions from base models and true labels.

        Parameters:
        - X_preds: Predictions from base models (n_samples x n_models)
        - y_true: Ground truth labels
        - model_type: One of ['logistic', 'random_forest', 'xgboost']
        - search_type: One of ['grid', 'random', 'bayesian']
        - scoring: Scikit-learn scoring metric
        - cv: Number of cross-validation folds
        - n_iter: Number of iterations for random search
        - bayes_trials: Number of trials for Bayesian optimization
        """
        model, param_grid = self._get_model_and_grid(model_type)
        X = X_preds.values if isinstance(X_preds, pd.DataFrame) else np.array(X_preds)

        search = self._create_search(model, param_grid, search_type, scoring, cv, n_iter, bayes_trials)
        search.fit(X, y_true)

        self.meta_model = search.best_estimator_
        self.best_params = search.best_params_
        self.fitted = True

    def predict(self, X_preds):
        """Returns hard predictions from the trained meta model."""
        if not self.fitted:
            raise RuntimeError("Meta model has not been fitted yet.")

        X = X_preds.values if isinstance(X_preds, pd.DataFrame) else np.array(X_preds)
        return self.meta_model.predict(X)

    def predict_proba(self, X_preds):
        """Returns predicted probabilities from the trained meta model."""
        if not self.fitted:
            raise RuntimeError("Meta model has not been fitted yet.")

        X = X_preds.values if isinstance(X_preds, pd.DataFrame) else np.array(X_preds)
        return self.meta_model.predict_proba(X)[:, 1]  # Binary classification

    def evaluate(self, X_preds, y_true):
        """
        Evaluates the model on F1, ROC AUC, accuracy, precision, recall
        """
        y_pred = self.predict(X_preds)
        y_score = self.predict_proba(X_preds)

        return {
                'Accuracy': accuracy_score(y_true, y_pred),
                'Precision': precision_score(y_true, y_pred),
                'Recall': recall_score(y_true, y_pred),
                'F1 Score': f1_score(y_true, y_pred),
                'ROC AUC': roc_auc_score(y_true, y_score)
        }


In [2]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, make_scorer
def evaluate_model(y_true, y_pred, y_proba):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred),
        'ROC AUC': roc_auc_score(y_true, y_proba)
    }

#### Load in LLM Model Outputs

In [21]:
# Load in the LLM outputs
mistral_pred = pd.read_excel('llm/prediction_dfmistral.xlsx')
llama_pred = pd.read_parquet('llm/prediction_dfllama.parquet', engine='pyarrow')

In [22]:
# REFACTORED MOST INTO UNSLOTH CODE
import json
import re

def parse_response(response):
    lines = response.splitlines()

    # Locate the start of JSON output
    line_num = next(i for i, line in enumerate(lines) if "Only return JSON" in line)
    json_lines = [line for i, line in enumerate(lines) if i > line_num]

    # Find where the JSON likely starts and ends
    try:
        start_json_index = next(i for i, line in enumerate(json_lines) if "{" in line)
        end_json_index = next(i for i, line in enumerate(json_lines) if "}" in line)
    except:
        print("Could not find JSON boundaries in response.")
        return response

    json_lines_final = json_lines[start_json_index:end_json_index + 1]
    json_text = ''.join(json_lines_final)

    # Remove hypothetical tokens
    json_text = re.sub(r'\(hypothetical\)', '', json_text)

    # Fix malformed related_topics list ending in }
    json_text = re.sub(
        r'"related_topics"\s*:\s*\[([^\]]*?)\}',  # matches until a closing brace
        lambda m: f'"related_topics": [{m.group(1).strip()}]',  # replace with proper array
        json_text
    )

    # Remove trailing commas before closing brackets/braces
    json_text = re.sub(r',\s*(\]|\})', r'\1', json_text)

    # Ensure all brackets and braces are balanced
    open_brackets = json_text.count('[')
    close_brackets = json_text.count(']')
    if open_brackets > close_brackets:
        json_text += ']' * (open_brackets - close_brackets)

    open_braces = json_text.count('{')
    close_braces = json_text.count('}')
    if open_braces > close_braces:
        json_text += '}' * (open_braces - close_braces)

    #print(json_text)
    final_json = json.loads(json_text)

    return final_json


def normalize_related_topics(val):
    if isinstance(val, list):
        return val
    elif pd.isnull(val):
        return None
    else:
        return [val] 

In [23]:
# Mistral outputs are fairly cleaned already
mistral_pred

,adverse_probability,financial_crime_relevance,related_topics
0,0.10,0.2,"['Tax Evasion', 'Economic Stability', 'Fiscal ..."
1,1.00,0.8,"['Sanctions Violations', 'Money Laundering', '..."
2,0.00,0.0,['Bribery and Corruption']
3,0.10,0.6,"['Money Laundering', 'AML Solutions']"
4,0.90,0.7,"['Sanctions Violations', 'Political Lobbying',..."
...,...,...,...
2995,1.00,1.0,"['Money Laundering', 'Terrorist Financing', 'F..."
2996,1.00,1.0,"['Money Laundering', 'Terrorist Financing', 'C..."
2997,0.80,1.0,['Fraud']
2998,0.00,0.0,[]


In [24]:
# Llama output needs cleaning, identify those index 
llama_problematic_index = llama_pred[~llama_pred['raw_output'].isna()].index

# Parse and clean
for ind in llama_problematic_index:
    tmp = parse_response(llama_pred.loc[ind,'raw_output'])
    if type(tmp) == 'dict':
        # We assign by extracting the values corresponding to keys in the dictionary
        llama_pred.loc[ind,'adverse_probability'] = tmp.get('adverse_probability')
        llama_pred.loc[ind,'financial_crime_relevance'] = tmp.get('financial_crime_relevance')
        llama_pred.loc[ind,'related_topics'] = tmp.get('related_topics')


Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.
Could not find JSON boundaries in response.


In [25]:
# Fillna with 0
llama_pred['adverse_probability'] = llama_pred['adverse_probability'].fillna(0)

In [26]:
# Export for further analysis
llama_pred.to_excel('llm/prediction_dfllama_cleaned.xlsx', index = False)

#### Loading the labelled data in

In [8]:
def split_data(df, label_col, random_state=42):
    """
    Splits dataset into training, validation, and test sets.
    """
    train_val_df, test_df = train_test_split(
        df, test_size=0.2, stratify=df[label_col], random_state=random_state
    )
    train_df, val_df = train_test_split(
        train_val_df, test_size=0.25, stratify=train_val_df[label_col], random_state=random_state
    )
    return train_df, val_df, test_df

In [9]:
# Load in the labelled data then do train test split (same one since we have set random seed to 42)
df = pd.read_excel("./sample_labelled/sample_labelled.xlsx", engine="openpyxl")
train_df, val_df, test_df = split_data(df, 'adverse_news_financial_crime_scandal_sanctions')

# Will do cross validation so we can have train_val 
train_val_df = pd.concat((train_df, val_df))

# Index is important
indx = train_val_df.index

#### Evaluation on LLM outputs Directly First

In [10]:
evaluate_model(y_true = df['adverse_news_financial_crime_scandal_sanctions'], y_pred = llama_pred['adverse_probability'].round(),
              y_proba = llama_pred['adverse_probability'])

{'Accuracy': 0.831,
 'Precision': 0.8629472407519709,
 'Recall': 0.835093896713615,
 'F1 Score': 0.8487921264539219,
 'ROC AUC': np.float64(0.8644873355358487)}

In [11]:
evaluate_model(y_true = df['adverse_news_financial_crime_scandal_sanctions'], y_pred = mistral_pred['adverse_probability'].round(),
              y_proba = mistral_pred['adverse_probability'])

{'Accuracy': 0.971,
 'Precision': 0.9850029994001199,
 'Recall': 0.9636150234741784,
 'F1 Score': 0.9741916345298132,
 'ROC AUC': np.float64(0.9932733618790934)}

#### Preparing for MetaClassifier Model

Compute similarity score for the embedding models 

In [12]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

class SimilarityScorer:
    def __init__(self, keywords, model_name="all-MiniLM-L12-v2"):
        """
        Initialize the similarity scorer with a given model and keyword list.
        
        Parameters:
        - keywords (List[str]): List of keywords to compare against.
        - model_name (str): Name of the SentenceTransformer model to load.
        """
        self.model = SentenceTransformer(model_name)
        self.keywords = keywords
        self.keyword_embeddings = self.model.encode(self.keywords, convert_to_tensor=True)

    def score(self, text, output_type='avg'):
        """
        Compute similarity score between input text and keyword embeddings.

        Parameters:
        - text (str): The input text.
        - output_type (str): Aggregation type for similarity ('avg' or 'max').

        Returns:
        - float: The aggregated similarity score.
        """
        text_embedding = self.model.encode(text, convert_to_tensor=True)
        similarities = util.cos_sim(text_embedding, self.keyword_embeddings)
        sim_array = similarities.cpu().numpy()

        if output_type == 'max':
            return float(np.max(sim_array))
        elif output_type == 'avg':
            return float(np.mean(sim_array))
        else:
            print("Unsupported output_type. Defaulting to 'avg'.")
            return float(np.mean(sim_array))

    def classify(self, text, threshold=0.5, output_type='avg'):
        """
        Binary classification based on similarity score threshold.

        Parameters:
        - text (str): Input text.
        - threshold (float): Similarity threshold to classify as relevant.
        - output_type (str): Aggregation type ('avg' or 'max').

        Returns:
        - int: 1 if similar to any keyword beyond threshold, else 0.
        """
        score = self.score(text, output_type)
        return int(score >= threshold)


C:\Users\angsi\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
# Initialize the scorer once
keywords = ["fraud", "money laundering", "corruption", "bribery",
            "sanction", "ponzi", "pyramid scheme",
            "insider trading", "terrorist financing", "tax-evasion"]

scorer = SimilarityScorer(keywords)

# Apply scoring using the scorer for train_val samples
similarity_scoring_df = pd.DataFrame()
similarity_scoring_df['cos_similarity_max'] = df.loc[indx, 'description'].apply(lambda x: scorer.score(x, output_type='max'))
similarity_scoring_df['cos_similarity_avg'] = df.loc[indx, 'description'].apply(lambda x: scorer.score(x, output_type='avg'))

# Apply scoring using scorer for test samples
similarity_scoring_test_df = pd.DataFrame()
similarity_scoring_test_df['cos_similarity_max'] = test_df['description'].apply(lambda x: scorer.score(x, output_type='max'))
similarity_scoring_test_df['cos_similarity_avg'] = test_df['description'].apply(lambda x: scorer.score(x, output_type='avg'))

In [14]:
# Preparing to do stacking
X_preds = pd.DataFrame({
    "llama": llama_pred.loc[indx,'adverse_probability'].tolist(),
    "mistral": mistral_pred.loc[indx,'adverse_probability'].tolist(),
    "keyword_similarity_max": similarity_scoring_df['cos_similarity_max'].tolist(),
    "keyword_similarity_avg": similarity_scoring_df['cos_similarity_avg'].tolist()
})
y_true = train_val_df.loc[indx, 'adverse_news_financial_crime_scandal_sanctions'].tolist()

ensemble = EnsembleMetaClassifier(base_model_names=X_preds.columns.tolist())
ensemble.fit(X_preds, y_true, model_type = 'xgboost')
metrics = ensemble.evaluate(X_preds, y_true)
metrics_df = pd.DataFrame([metrics])
metrics_df

C:\Users\angsi\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:45:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\angsi\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:45:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\angsi\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:45:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\angsi\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:45:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtr

,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,0.990417,0.983405,1.0,0.991633,0.995788


In [16]:
#Evaluate Test Results
test_indx = test_df.index
X_preds_test = pd.DataFrame({
    "llama": llama_pred.loc[test_indx,'adverse_probability'].tolist(),
    "mistral": mistral_pred.loc[test_indx,'adverse_probability'].tolist(),
    "keyword_similarity_max": similarity_scoring_test_df['cos_similarity_max'].tolist(),
    "keyword_similarity_avg": similarity_scoring_test_df['cos_similarity_avg'].tolist()
})
y_true_test = df.loc[test_indx, 'adverse_news_financial_crime_scandal_sanctions'].tolist()
evaluate_model(y_true = y_true_test, y_pred = ensemble.predict(X_preds_test), y_proba = ensemble.predict_proba(X_preds_test))

{'Accuracy': 0.9883333333333333,
 'Precision': 0.9798850574712644,
 'Recall': 1.0,
 'F1 Score': 0.9898403483309144,
 'ROC AUC': np.float64(0.9917118626796046)}